# Day 4 Assignments

This notebook contains all four data-analysis assignments in one file.
Run each section individually to view the output.

In [1]:
from pathlib import Path
import pandas as pd

base_dir = Path.cwd()
required_files = [
    'patient_clinical_data_raw.csv',
    'ecommerce_orders_raw.csv',
    'iot_sensor_data_raw.csv',
    'customers.csv',
    'products.csv',
    'orders.csv'
]
if not all((base_dir / name).exists() for name in required_files):
    base_dir = base_dir.parent

base_dir

PosixPath('/home/user01/Assignments/Assignment Day-4')

## Assignment 3: IoT Sensor Data

Analyze device status, battery health, and abnormal readings.

In [2]:
sensor = pd.read_csv(base_dir / 'iot_sensor_data_raw.csv')
sensor['Timestamp'] = pd.to_datetime(sensor['Timestamp'])
sensor = sensor.sort_values('Timestamp')
print('Shape:', sensor.shape)
print('\nMissing values:')
print(sensor.isnull().sum())

for col in ['Temperature', 'Humidity', 'Pressure', 'Vibration', 'Battery_Level']:
    sensor[col] = pd.to_numeric(sensor[col], errors='coerce')
    sensor[col] = sensor[col].fillna(sensor[col].median())

sensor['Battery_Status'] = pd.cut(
    sensor['Battery_Level'],
    bins=[-1, 19, 49, float('inf')],
    labels=['Critical', 'Moderate', 'Healthy'],
    right=False
)
sensor['Temperature_Status'] = pd.cut(
    sensor['Temperature'],
    bins=[-1, 85, 95, float('inf')],
    labels=['Normal', 'Warning', 'Critical'],
    right=False
)
sensor['Vibration_Status'] = pd.cut(
    sensor['Vibration'],
    bins=[-1, 3, 5, float('inf')],
    labels=['Normal', 'Warning', 'Critical'],
    right=False
)

conditions = [
    (sensor['Temperature_Status'].eq('Critical')) | (sensor['Vibration_Status'].eq('Critical')) | (sensor['Battery_Status'].eq('Critical')) ,
    (sensor['Temperature_Status'].eq('Warning')) | (sensor['Vibration_Status'].eq('Warning')) | (sensor['Battery_Status'].eq('Moderate'))
]
sensor['Machine_Health'] = 'Normal'
sensor.loc[conditions[0], 'Machine_Health'] = 'Critical'
sensor.loc[conditions[1], 'Machine_Health'] = 'Warning'

print('\nAverage temperature by device:')
print(sensor.groupby('Device_ID')['Temperature'].mean())
print('\nMachine health counts:')
print(sensor['Machine_Health'].value_counts())
print('\nHighest maintenance priority device:')
summary = sensor.groupby('Device_ID')['Machine_Health'].apply(lambda x: (x.isin(['Warning', 'Critical']).mean() * 100))
print(summary.sort_values(ascending=False).head())

Shape: (20000, 9)

Missing values:
Timestamp           0
Device_ID           0
Temperature       299
Humidity          300
Pressure          300
Vibration         298
Battery_Level       0
Location            0
Machine_Status      0
dtype: int64

Average temperature by device:
Device_ID
DEV_001    70.248192
DEV_002    70.370555
DEV_003    69.935013
DEV_004    70.576572
DEV_005    70.116720
DEV_006    70.006095
DEV_007    70.145846
DEV_008    70.315545
Name: Temperature, dtype: float64

Machine health counts:
Machine_Health
Warning     10511
Normal       7929
Critical     1560
Name: count, dtype: int64

Highest maintenance priority device:
Device_ID
DEV_006    62.311762
DEV_008    60.337214
DEV_004    60.231815
DEV_001    60.221870
DEV_002    60.118577
Name: Machine_Health, dtype: float64
